### Structured Output

Structured output allows you to request a model to return its response in a **specific format or schema**.This is useful when you want the model's output to be **consistent, predictable, and easy to parse or use in further processing**.LangChain supports different schema types and methods to enforce structured output.

In [9]:
from langchain.chat_models import init_chat_model
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15'}}, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000025BF764A060>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000025BF764A9E0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [77]:
import os
from langchain.chat_models import init_chat_model
api_key=os.getenv("GROQ_API_KEY")
model=init_chat_model(
    "groq:qwen/qwen3.6-27b",
    api_key=api_key
     )
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15'}}, client=<groq.resources.chat.completions.Completions object at 0x0000025BFF1E4F50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000025BFF1E5220>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

### Pydantic

Pydantic is a Python library used to define and validate the structure of data using Python type hints.

Pydantic models provide features such as **field validation, descriptions, default values, and nested structures**, making them useful for defining structured data.

BaseModel
→ Creates a structured data model and validates the data. It validates data when you create an instance of the model.

Field
→ Defines a field and lets you add extra settings such as validation rules, default values, and metadata.

description
→ Explains what that field means, helping developers, documentation, APIs, and AI models understand what data should go there.

In [21]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

Difference

- Without BaseModel → Type hints are mostly just annotations; no automatic validation.

- With BaseModel    → Pydantic validates the input data and raises an error if it doesn't match the expected structure/type.

In [22]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15'}}, client=<groq.resources.chat.completions.Completions object at 0x0000025BF7681F20>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000025BF76827A0>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'function': {'name': 'Mo

In [23]:
response=model.invoke("Provide details about a movie Mersal")
print(response.content)


<think>
Here's a thinking process:

1.  **Understand User Request:** The user wants details about the movie "Mersal".
2.  **Identify Key Information Needed:** 
   - Title: Mersal
   - Release Year
   - Director
   - Producer(s)
   - Cast (Main actors/actresses)
   - Music Director
   - Genre
   - Plot Summary (brief, non-spoilery)
   - Box Office Performance
   - Awards/Reception
   - Notable Facts/Trivia
   - Language/Remakes (if any)
3.  **Retrieve Knowledge (Internal Training Data):**
   - *Title:* Mersal
   - *Release Year:* 2017 (August 15, 2017)
   - *Director:* Atlee Kumar Bala
   - *Producer:* A. R. Murugadoss (under A. R. Murugadoss Films)
   - *Cast:* Vijay, Samantha Ruth Prabhu, Kajal Aggarwal, Shruti Haasan, Sanjay Dutt, Prabhu, Prakash Raj, Soori, Aashiq Abu, etc.
   - *Music:* Harris Jayaraj
   - *Cinematography:* Ravi K. Chandran
   - *Editing:* A. Sreekar Prasad
   - *Genre:* Action drama, thriller
   - *Plot:* Centers around three brothers (Mers = M, E, R, S, A, L - a

In [26]:
response=model_with_structure.invoke("Provide details about the moview Mersal")
response

Movie(title='Mersal', year=2017, director='Atlee', rating=6.0)

In [43]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """Movie details"""
    title: str = Field(...,description="The title of the movie") # ... means the field is required.
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's imbd rating")

model_with_structure=model.with_structured_output(Movie,include_raw=True) 
# with_structured_output -> Tells the model to return its response in the structure defined by Movie.
# include_raw=True  → raw response + parsed output + parsing error
# include_raw=False → only the parsed structured output

response=model_with_structure.invoke("Provide details about kaththi movie")
print(response)

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: "Provide details about kaththi movie"\n   - Key entity: "Kaththi" (a movie)\n   - Need to provide details about it.\n\n2.  **Identify Required Information for Tool:**\n   - The available tool is `Movie` which requires: `title`, `year`, `director`, `rating` (all required).\n   - I need to find these details for the movie "Kaththi".\n\n3.  **Knowledge Retrieval (Internal):**\n   - Movie: Kaththi\n   - Year: 2014\n   - Director: A. L. Vijay\n   - IMDb Rating: Around 7.3 (I should verify or use a reasonable approximation, but I\'ll stick to known data: IMDb rating is 7.3/10)\n   - Let me double-check: Kaththi (2014) directed by A. L. Vijay, starring Vijay. IMDb rating is indeed around 7.3.\n\n4.  **Formulate Tool Call:**\n   - `title`: "Kaththi"\n   - `year`: 2014\n   - `director`: "A. L. Vijay"\n   - `rating`: 7.3\n\n   Wait, the tool definitio

In [47]:
# include_raw=True  → raw response + parsed output + parsing error
print(response["raw"])
print(response["parsed"])
print(response["parsing_error"])

content='' additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: "Provide details about kaththi movie"\n   - Key entity: "Kaththi" (a movie)\n   - Need to provide details about it.\n\n2.  **Identify Required Information for Tool:**\n   - The available tool is `Movie` which requires: `title`, `year`, `director`, `rating` (all required).\n   - I need to find these details for the movie "Kaththi".\n\n3.  **Knowledge Retrieval (Internal):**\n   - Movie: Kaththi\n   - Year: 2014\n   - Director: A. L. Vijay\n   - IMDb Rating: Around 7.3 (I should verify or use a reasonable approximation, but I\'ll stick to known data: IMDb rating is 7.3/10)\n   - Let me double-check: Kaththi (2014) directed by A. L. Vijay, starring Vijay. IMDb rating is indeed around 7.3.\n\n4.  **Formulate Tool Call:**\n   - `title`: "Kaththi"\n   - `year`: 2014\n   - `director`: "A. L. Vijay"\n   - `rating`: 7.3\n\n   Wait, the tool definition says it\'s for "M

## Nested Structure

In [53]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None, description="Budget in INR")

model_with_structure=model.with_structured_output(MovieDetails,include_raw=True)

response=model_with_structure.invoke("Provide details about the tamil movie Villu")
response["parsed"]

MovieDetails(title='Villu', year=2009, cast=[Actor(name='Arya', role='Inspector Shiva'), Actor(name='Trisha Krishnan', role='Pooja'), Actor(name='Nassar', role='Police Commissioner'), Actor(name='Jaggappathi Babu', role='Dhanraj'), Actor(name='Vivek', role='PC Vijay')], genres=['Action', 'Thriller', 'Crime'], budget=None)

### TypedDict

`TypedDict` is used to define the **expected structure and data types of a Python dictionary**.

```python
from typing import TypedDict

class Movie(TypedDict):
    title: str
    year: int
    rating: float
```

The expected dictionary structure is:

```python
movie: Movie = {
    "title": "Inception",
    "year": 2010,
    "rating": 8.8
}
```

### Key Point

`TypedDict` mainly provides **type hints and structure** for dictionaries.

```text
TypedDict
→ "Your dictionary should look like this."
```

It does **not automatically validate data at runtime**:

```python
movie: Movie = {
    "title": "Inception",
    "year": "2010"  # Python can still run this
}
```

### TypedDict vs Pydantic BaseModel

#### TypedDict

* Defines the expected structure of a dictionary.
* Mainly used for type hints and static type checking.
* Does not automatically validate data at runtime.
* Access values using `movie["title"]`.
* By default, all fields are required.

To explicitly mark a field as required, use `Required`:

```python
class MovieDict(TypedDict):
    title: Required[str]
```

To make a field optional, use `NotRequired`:

```python
class MovieDict(TypedDict):
    title: str
    year: int
    director: NotRequired[str]
```

---

#### Pydantic BaseModel

* Defines the structure of data.
* Automatically validates data at runtime.
* Supports descriptions, default values, and custom validation.
* Access values using `movie.title`.
* A field **without a default value is required**:

```python
class Movie(BaseModel):
    title: str
```

* A field with a default value is **not required**:

```python
class Movie(BaseModel):
    title: str = "Unknown"
```

* `Field(...)` can explicitly mark a field as required while also allowing extra metadata:

```python
class Movie(BaseModel):
    title: str = Field(
        ...,
        description="The title of the movie"
    )
```


In [67]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, "The title of the movie"]
    year: Annotated[int,  "The year the movie was released"]
    director: Annotated[str , "The director of the movie"]
    rating: Annotated[float , "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [78]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'The Avengers',
 'year': 2012}

In [ ]:
model.profile # Profile is a property of the LangChain model object which gives  metadata about the model's capabilities.

{'name': 'Gemini 3.6 Flash',
 'release_date': '2026-07-21',
 'last_updated': '2026-07-21',
 'open_weights': False,
 'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True,
 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'],
 'reasoning_effort_default': 'medium'}

## Dataclass
### Dataclass vs TypedDict vs Pydantic BaseModel

#### `TypedDict`

`TypedDict` is used to define the **expected structure of a dictionary**.

* Mainly used for type hints and static type checking.
* The actual data remains a normal Python `dict`.
* Does not automatically validate data at runtime.
* Access values using `user["name"]`.

```python
from typing import TypedDict

class User(TypedDict):
    name: str
    age: int

user = {
    "name": "Aakash",
    "age": 22
}
```

---

#### `@dataclass`

A `dataclass` is used to create a **simple Python object for storing data**.

* Built into Python.
* Automatically generates methods such as `__init__`, `__repr__`, and `__eq__`.
* Does not automatically validate type hints at runtime.
* Access values using `user.name`.

```python
from dataclasses import dataclass

@dataclass
class User:
    name: str
    age: int

user = User(name="Aakash", age=22)
```

---

#### Pydantic `BaseModel`

Pydantic `BaseModel` creates a **structured data object with runtime validation**.

* Automatically validates data when creating the object.
* Supports type conversion, default values, descriptions, custom validation, serialization, and schema generation.
* Provides functionality similar to `__init__`, `__repr__`, and `__eq__`.
* Access values using `user.name`.

```python
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int

user = User(name="Aakash", age=22)
```

---



In [79]:
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [83]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

## Difference in Single Line

- **Pydantic** → Use when you need **runtime validation and parsing of external/input data** (APIs, JSON, user input).

- **Dataclass** → Use when you need a **simple, lightweight Python object to represent trusted internal data**.

- **TypedDict** → Use when you want a **normal dictionary with static type definitions for type checking and IDE support**.